In [53]:
import importlib
import utils
importlib.reload(utils)

import numpy as np

In [54]:
# ==================
# GENERAL PARAMETERS
# ==================

n = 5
steps = 12               # number of steps required for time evolution

def u(x):
    return np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)

nu = 1e-3                 # diffusion coefficient 
save_every = 200          # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1                 # controls time step relative to grid spacing. affects stability of time0-step scheme



# =============
# TN PARAMETERS
# =============

# parameters for forming the initial MPS
init_cutoff = 1e-12
init_max_bond = 64

# parameters for forming the initial MPO
mpo_cutoff = 1e-12
mpo_max_bond = 256

In [55]:
N     = 2**n
x     = np.linspace(0, 1, N, endpoint=False)

dx    = x[1] - x[0] 
dt    = cfl * dx*dx / nu 
u0    = u(x)
alpha = nu * dt / (dx * dx)

The result below show that when converted to matrices, the MPOs are almost exactly the same.

In [56]:
L = utils.laplacian_1d(N, dx, "dirichlet", "dense")

A_dense = np.eye(N) + dt * nu * L                  # your dense time-step matrix
A_svd = utils.mat_to_qtt_mpo(A_dense, n)
A_manual = utils.qtt_diffusion_mpo(n, alpha)

A_svd_dense = np.asarray(A_svd.to_dense()).reshape(N, N)
A_manual_dense = np.asarray(A_manual.to_dense()).reshape(N, N)

print("||A_svd - A_dense|| =", np.linalg.norm(A_svd_dense - A_dense))
print("||A_manual - A_dense|| =", np.linalg.norm(A_manual_dense - A_dense))
print("||A_manual - A_svd||  =", np.linalg.norm(A_manual_dense - A_svd_dense))

||A_svd - A_dense|| = 4.27863164248793e-15
||A_manual - A_dense|| = 1.6014269617894046e-15
||A_manual - A_svd||  = 4.4258996490073184e-15


The two code cells below show that the manual MPO explodes in bond dimension very quickly, which leads to much longer computation times.

In [57]:
mps0 = utils.vec_to_qtt_mps(u0, n)
saved, bonds = utils.evolve_mps_timed(mps0, A_manual, steps)

step  0: 0.001085 s, max bond = 12
step  1: 0.004791 s, max bond = 36
step  2: 0.045140 s, max bond = 64
step  3: 0.092466 s, max bond = 64
step  4: 0.229970 s, max bond = 64
step  5: 0.341642 s, max bond = 64
step  6: 1.094712 s, max bond = 64
step  7: 1.627863 s, max bond = 64
step  8: 3.597494 s, max bond = 64
step  9: 10.395433 s, max bond = 64
step 10: 30.613778 s, max bond = 64
step 11: 101.448812 s, max bond = 64


In [58]:
mps0 = utils.vec_to_qtt_mps(u0, n)
saved, bonds = utils.evolve_mps_timed(mps0, A_svd, steps)

step  0: 0.003373 s, max bond = 4
step  1: 0.000787 s, max bond = 4
step  2: 0.000720 s, max bond = 4
step  3: 0.000704 s, max bond = 4
step  4: 0.000689 s, max bond = 4
step  5: 0.000689 s, max bond = 4
step  6: 0.000684 s, max bond = 4
step  7: 0.000681 s, max bond = 4
step  8: 0.000703 s, max bond = 4
step  9: 0.000718 s, max bond = 4
step 10: 0.000712 s, max bond = 4
step 11: 0.001501 s, max bond = 4


Below I take a look at the tensor shape and realise that they are quite different.

In [59]:
print("A_manual bond sizes:", A_manual.bond_sizes())
print("A_svd bond sizes:", A_svd.bond_sizes())


for i, T in enumerate(A_manual):
    print(f"A_manual tensor {i}: shape = {T.shape}")

for i, T in enumerate(A_svd):
    print(f"A_svd tensor {i}: shape = {T.shape}")

A_manual bond sizes: [5, 5, 5, 5, 3]
A_svd bond sizes: [3, 3, 3, 3]
A_manual tensor 0: shape = (3, 5, 2, 2)
A_manual tensor 1: shape = (5, 5, 2, 2)
A_manual tensor 2: shape = (5, 5, 2, 2)
A_manual tensor 3: shape = (5, 5, 2, 2)
A_manual tensor 4: shape = (5, 3, 2, 2)
A_svd tensor 0: shape = (2, 2, 3)
A_svd tensor 1: shape = (3, 2, 2, 3)
A_svd tensor 2: shape = (3, 2, 2, 3)
A_svd tensor 3: shape = (3, 2, 2, 3)
A_svd tensor 4: shape = (3, 2, 2)


Even after compression, the manual MPO remains the same.

In [60]:
A_manual.compress(cutoff=1e-12)
print("A_manual bond sizes:", A_manual.bond_sizes())
for i, T in enumerate(A_manual):
    print(f"A_manual tensor {i}: shape = {T.shape}")

A_manual bond sizes: [5, 5, 5, 5, 3]
A_manual tensor 0: shape = (3, 5, 2, 2)
A_manual tensor 1: shape = (5, 5, 2, 2)
A_manual tensor 2: shape = (5, 5, 2, 2)
A_manual tensor 3: shape = (5, 5, 2, 2)
A_manual tensor 4: shape = (5, 3, 2, 2)


Conclusion: `from.dense` optimises the MPO in such a way that my manual construction is not able to emulate